# Bayesian Hyperparameter Tuning for Conventional Stacking Models

This notebook performs Bayesian hyperparameter tuning for 7 conventional stacking configurations. The tuning includes base learners and the Logistic Regression meta-learner.


In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path

# Reproducibility
RANDOM_STATE = 42
N_SPLITS = 5

# Before running this notebook:
# 1) Place the cleaned and outlier-processed dataset in the Data folder.
# 2) Update DATA_PATH if your local file name or location is different.
DATA_PATH = Path("../Data/Partition class 0 Five_5 cluster on cleaned_data_afterDuplicate_tranform_dummies_numerical_STD_drop_Outlier.csv")
TARGET_COLUMN = "HeartDisease"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}\n"
        "Please place the cleaned dataset in the Data folder or update DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)

X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

print("Dataset shape:", df.shape)
print("Feature matrix:", X.shape)
print("Target distribution:")
print(y.value_counts())


# Tuning LR+ET Conventional using Bayesian optimization

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
from imblearn.combine import SMOTEENN

import optuna
from optuna.samplers import TPESampler


# =========================================================
#   lr_C, lr_max_iter
#   et_n_estimators, et_max_depth, et_min_samples_split, et_min_samples_leaf,
#   et_max_features, et_bootstrap
#   meta_C, meta_max_iter
# =========================================================
def build_stacking_from_params(params_joint):
    lr_pipe = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=params_joint["lr_C"],
            max_iter=params_joint["lr_max_iter"],
            random_state=RANDOM_STATE
        )
    )

    et_pipe = make_pipeline(
        StandardScaler(),
        ExtraTreesClassifier(
            n_estimators=params_joint["et_n_estimators"],
            max_depth=params_joint["et_max_depth"],
            min_samples_split=params_joint["et_min_samples_split"],
            min_samples_leaf=params_joint["et_min_samples_leaf"],
            max_features=params_joint["et_max_features"],
            bootstrap=params_joint["et_bootstrap"],
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    )

    # meta-learner: Logistic Regression (no scaling inside Stacking, so we scale in base learners,
    # meta sees their predictions)
    final_lr = LogisticRegression(
        C=params_joint["meta_C"],
        max_iter=params_joint["meta_max_iter"],
        random_state=RANDOM_STATE
    )

    stack_model = StackingClassifier(
        estimators=[
            ('LR', lr_pipe),
            ('ET', et_pipe)
        ],
        final_estimator=final_lr,
        cv=5,
        passthrough=False,
        n_jobs=-1
    )

    return stack_model


# =========================================================
# =========================================================
def evaluate_joint_params_inner_cv(
    X_train_outer,
    y_train_outer,
    params_joint,
    n_splits_inner=3,
    random_state_inner=123
):
   """
Inner cross-validation for Bayesian optimization.
SMOTEENN is applied only to the inner training set.
The objective is to maximize mean recall.
"""
    kf_inner = StratifiedKFold(
        n_splits=n_splits_inner,
        shuffle=True,
        random_state=random_state_inner
    )

    recalls = []

    for inner_tr_idx, inner_val_idx in kf_inner.split(X_train_outer, y_train_outer):
        X_tr, X_val = (
            X_train_outer.iloc[inner_tr_idx],
            X_train_outer.iloc[inner_val_idx],
        )
        y_tr, y_val = (
            y_train_outer.iloc[inner_tr_idx],
            y_train_outer.iloc[inner_val_idx],
        )

        # ===== Oversample inner-train =====
        smoteenn = SMOTEENN(random_state=42)
        X_tr_res, y_tr_res = smoteenn.fit_resample(X_tr, y_tr)

        # ===== train stacking params_joint =====
        model = build_stacking_from_params(params_joint)
        model.fit(X_tr_res, y_tr_res)

        # ===== Predict inner-val ( oversample) =====
        y_val_pred = model.predict(X_val)

        # ===== Metric: recall =====
        rec = recall_score(y_val, y_val_pred, zero_division=0)
        recalls.append(rec)

    return np.mean(recalls)



# =========================================================
def tune_joint_params_for_outer_fold(
    X_train_outer,
    y_train_outer,
    n_trials=30,
    n_splits_inner=3,
    random_state_inner=123,
    seed_optuna=42
):
    """
     Optuna :
    - Logistic Regression base
    - ExtraTrees base
    - Meta Logistic Regression
     (joint)
     Objective: mean recall from inner CV.
    """

    def objective(trial):
        # -------- Base LR params --------
        lr_C = trial.suggest_float("lr_C", 1e-4, 1e2, log=True)
        lr_max_iter = trial.suggest_int("lr_max_iter", 100, 2000)

        # -------- ExtraTrees params --------
        et_n_estimators = trial.suggest_int("et_n_estimators", 50, 300)
        et_max_depth = trial.suggest_int("et_max_depth", 3, 50)
        et_min_samples_split = trial.suggest_int("et_min_samples_split", 2, 20)
        et_min_samples_leaf = trial.suggest_int("et_min_samples_leaf", 1, 10)
        et_max_features = trial.suggest_categorical(
            "et_max_features", ["sqrt", "log2", None]
        )
        et_bootstrap = trial.suggest_categorical(
            "et_bootstrap", [True, False]
        )

        # -------- Meta LR params --------
        meta_C = trial.suggest_float("meta_C", 1e-4, 1e2, log=True)
        meta_max_iter = trial.suggest_int("meta_max_iter", 100, 2000)

        params_joint = {
            "lr_C": lr_C,
            "lr_max_iter": lr_max_iter,

            "et_n_estimators": et_n_estimators,
            "et_max_depth": et_max_depth,
            "et_min_samples_split": et_min_samples_split,
            "et_min_samples_leaf": et_min_samples_leaf,
            "et_max_features": et_max_features,
            "et_bootstrap": et_bootstrap,

            "meta_C": meta_C,
            "meta_max_iter": meta_max_iter
        }

        mean_recall = evaluate_joint_params_inner_cv(
            X_train_outer,
            y_train_outer,
            params_joint,
            n_splits_inner=n_splits_inner,
            random_state_inner=random_state_inner
        )

        # maximize recall
        return mean_recall

    study = optuna.create_study(
        direction="maximize",
        sampler=TPESampler(seed=seed_optuna),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    best_params_joint = study.best_params.copy()
    best_inner_recall = study.best_value  # Mean recall from inner CV.

    return best_params_joint, best_inner_recall


# =========================================================
# MAIN: Outer nested CV and fold-level reporting.
# =========================================================
def nested_cv_pipeline_stacking_joint(
    X,
    y,
    outer_splits=5,
    inner_splits=3,
    n_trials=30,
    seed_outer=42,
    seed_inner=123,
    seed_optuna=42
):
    """
    outer_splits: Number of folds for final evaluation (e.g., 5).
    inner_splits: Number of folds for tuning (e.g., 3).
    n_trials:        Optuna  outer fold
    """

    outer_kf = StratifiedKFold(
        n_splits=outer_splits,
        shuffle=True,
        random_state=seed_outer
    )

    fold_metrics = []
    fold_params = []

    print("===== Nested CV Stacking (joint Bayesian tuning for LR + ET + meta-LR) =====\n")

    for fold_idx, (tr_idx, te_idx) in enumerate(outer_kf.split(X, y), start=1):
        X_train_outer = X.iloc[tr_idx] if hasattr(X, "iloc") else X[tr_idx]
        X_test_outer  = X.iloc[te_idx] if hasattr(X, "iloc") else X[te_idx]
        y_train_outer = y.iloc[tr_idx] if hasattr(y, "iloc") else y[tr_idx]
        y_test_outer  = y.iloc[te_idx] if hasattr(y, "iloc") else y[te_idx]

        # -------------------------------------------------
        # -------------------------------------------------
        best_params_fold, best_inner_recall = tune_joint_params_for_outer_fold(
            X_train_outer,
            y_train_outer,
            n_trials=n_trials,
            n_splits_inner=inner_splits,
            random_state_inner=seed_inner,
            seed_optuna=seed_optuna + fold_idx
        )

        fold_params.append(best_params_fold)

        # -------------------------------------------------
        # -------------------------------------------------
        smoteenn = SMOTEENN(random_state=RANDOM_STATE)
        X_train_res, y_train_res = smoteenn.fit_resample(X_train_outer, y_train_outer)

        model_final = build_stacking_from_params(best_params_fold)
        model_final.fit(X_train_res, y_train_res)

        y_pred = model_final.predict(X_test_outer)
        y_proba = model_final.predict_proba(X_test_outer)[:, 1]

        acc  = accuracy_score(y_test_outer, y_pred)
        prec = precision_score(y_test_outer, y_pred, zero_division=0)
        rec  = recall_score(y_test_outer, y_pred, zero_division=0)
        f1   = f1_score(y_test_outer, y_pred, zero_division=0)
        auc  = roc_auc_score(y_test_outer, y_proba)

        cm = confusion_matrix(y_test_outer, y_pred)

        fold_metrics.append({
            "fold": fold_idx,
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "f1": f1,
            "auc": auc,
            "TN": int(cm[0,0]),
            "FP": int(cm[0,1]),
            "FN": int(cm[1,0]),
            "TP": int(cm[1,1]),
            "best_inner_recall": best_inner_recall
        })

        # -------------------------------------------------
        # 3) Display fold-level results.
        # -------------------------------------------------
        print(f"[Outer Fold {fold_idx}]")
        print(f"  Inner-CV tuned recall (objective): {best_inner_recall:.4f}")
        print(f"  Test Performance on this fold:")
        print(f"    Acc={acc:.4f}  Prec={prec:.4f}  Rec={rec:.4f}  "
              f"F1={f1:.4f}  AUC={auc:.4f}")
        print("  Confusion Matrix:")
        print(cm)
        print("  Best Joint Params:")
        print(best_params_fold)
        print("-"*100)

    # -------------------------------------------------
    # 4) Summarize results across all outer folds.
    # -------------------------------------------------
    df_res = pd.DataFrame(fold_metrics)

    print("\n===== Summary over all outer folds =====")
    for m in ["accuracy", "precision", "recall", "f1", "auc"]:
        mean_val = df_res[m].mean()
        std_val  = df_res[m].std(ddof=1)
        print(f"{m.capitalize():<10} mean={mean_val:.4f}  std={std_val:.4f}")

    print("\nPer-fold confusion / best params:")
    for row in fold_metrics:
        print(f" Fold {row['fold']}: "
              f"TN={row['TN']} FP={row['FP']} FN={row['FN']} TP={row['TP']}")
        print(f"  Params: {fold_params[row['fold']-1]}")
        print()

    summary_results = {
        "Accuracy":  (df_res["accuracy"].mean(),  df_res["accuracy"].std(ddof=1)),
        "Precision": (df_res["precision"].mean(), df_res["precision"].std(ddof=1)),
        "Recall":    (df_res["recall"].mean(),    df_res["recall"].std(ddof=1)),
        "F1":        (df_res["f1"].mean(),        df_res["f1"].std(ddof=1)),
        "AUC":       (df_res["auc"].mean(),       df_res["auc"].std(ddof=1)),
    }

    return summary_results, fold_metrics, fold_params


# =========================================================
# Usage:
#X = df.drop("target", axis=1)
# y = df["target"]
# Then call:
#
summary_results, fold_metrics, params_each_fold = nested_cv_pipeline_stacking_joint(
     X, y,
     outer_splits=5,
     inner_splits=3,
     n_trials=100,
     seed_outer=42,
     seed_inner=123,
     seed_optuna=42
 )
#
#fold_metrics     -> fold-level performance and confusion matrix. counts
# =========================================================


# Tuning RF+ET Conventional using Bayesian optimization

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
from imblearn.combine import SMOTEENN

import optuna
from optuna.samplers import TPESampler


# =========================================================
#   rf_n_estimators, rf_max_depth, rf_min_samples_split,
#   rf_min_samples_leaf, rf_max_features, rf_bootstrap
#
#   et_n_estimators, et_max_depth, et_min_samples_split,
#   et_min_samples_leaf, et_max_features, et_bootstrap
#
#   meta_C, meta_max_iter
# =========================================================
def build_stacking_from_params(params_joint):
    rf_pipe = make_pipeline(
        StandardScaler(),
        RandomForestClassifier(
            n_estimators=params_joint["rf_n_estimators"],
            max_depth=params_joint["rf_max_depth"],
            min_samples_split=params_joint["rf_min_samples_split"],
            min_samples_leaf=params_joint["rf_min_samples_leaf"],
            max_features=params_joint["rf_max_features"],
            bootstrap=params_joint["rf_bootstrap"],
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    )

    et_pipe = make_pipeline(
        StandardScaler(),
        ExtraTreesClassifier(
            n_estimators=params_joint["et_n_estimators"],
            max_depth=params_joint["et_max_depth"],
            min_samples_split=params_joint["et_min_samples_split"],
            min_samples_leaf=params_joint["et_min_samples_leaf"],
            max_features=params_joint["et_max_features"],
            bootstrap=params_joint["et_bootstrap"],
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    )

    # meta-learner: Logistic Regression
    final_lr = LogisticRegression(
        C=params_joint["meta_C"],
        max_iter=params_joint["meta_max_iter"],
        random_state=RANDOM_STATE
    )

    stack_model = StackingClassifier(
        estimators=[
            ('RF', rf_pipe),
            ('ET', et_pipe)
        ],
        final_estimator=final_lr,
        cv=5,
        passthrough=False,
        n_jobs=-1
    )

    return stack_model


# =========================================================
# =========================================================
def evaluate_joint_params_inner_cv(
    X_train_outer,
    y_train_outer,
    params_joint,
    n_splits_inner=3,
    random_state_inner=123
):
    """
     training set  outer fold
     inner CV ( 3-fold)
     inner split:
        - SMOTEENN  inner-train
        -  stacking  params_joint
        - predict inner-val
     mean recall  maximize  tuning
    """
    kf_inner = StratifiedKFold(
        n_splits=n_splits_inner,
        shuffle=True,
        random_state=random_state_inner
    )

    recalls = []

    for inner_tr_idx, inner_val_idx in kf_inner.split(X_train_outer, y_train_outer):
        X_tr, X_val = (
            X_train_outer.iloc[inner_tr_idx],
            X_train_outer.iloc[inner_val_idx],
        )
        y_tr, y_val = (
            y_train_outer.iloc[inner_tr_idx],
            y_train_outer.iloc[inner_val_idx],
        )

        # ===== Oversample only the inner-training set. =====
        smoteenn = SMOTEENN(random_state=42)
        X_tr_res, y_tr_res = smoteenn.fit_resample(X_tr, y_tr)

        # ===== stacking params_joint =====
        model = build_stacking_from_params(params_joint)
        model.fit(X_tr_res, y_tr_res)

        # ===== Predict inner-val ( oversample) =====
        y_val_pred = model.predict(X_val)

        # ===== Metric: recall =====
        rec = recall_score(y_val, y_val_pred, zero_division=0)
        recalls.append(rec)

    return np.mean(recalls)


# =========================================================
# Return best_params_joint and best_inner_recall.
# =========================================================
def tune_joint_params_for_outer_fold(
    X_train_outer,
    y_train_outer,
    n_trials=30,
    n_splits_inner=3,
    random_state_inner=123,
    seed_optuna=42
):
    """
     Optuna :
    - RandomForest base
    - ExtraTrees base
    - Meta Logistic Regression
     (joint tuning)
     Objective: mean recall from inner CV.
    """

    def objective(trial):
        # -------- RandomForest params --------
        rf_n_estimators = trial.suggest_int("rf_n_estimators", 50, 300)
        rf_max_depth = trial.suggest_int("rf_max_depth", 3, 50)
        rf_min_samples_split = trial.suggest_int("rf_min_samples_split", 2, 20)
        rf_min_samples_leaf = trial.suggest_int("rf_min_samples_leaf", 1, 10)
        rf_max_features = trial.suggest_categorical(
            "rf_max_features", ["sqrt", "log2", None]
        )
        rf_bootstrap = trial.suggest_categorical(
            "rf_bootstrap", [True, False]
        )

        # -------- ExtraTrees params --------
        et_n_estimators = trial.suggest_int("et_n_estimators", 50, 300)
        et_max_depth = trial.suggest_int("et_max_depth", 3, 50)
        et_min_samples_split = trial.suggest_int("et_min_samples_split", 2, 20)
        et_min_samples_leaf = trial.suggest_int("et_min_samples_leaf", 1, 10)
        et_max_features = trial.suggest_categorical(
            "et_max_features", ["sqrt", "log2", None]
        )
        et_bootstrap = trial.suggest_categorical(
            "et_bootstrap", [True, False]
        )

        # -------- Meta LR params --------
        meta_C = trial.suggest_float("meta_C", 1e-4, 1e2, log=True)
        meta_max_iter = trial.suggest_int("meta_max_iter", 100, 2000)

        params_joint = {
            # RF base
            "rf_n_estimators": rf_n_estimators,
            "rf_max_depth": rf_max_depth,
            "rf_min_samples_split": rf_min_samples_split,
            "rf_min_samples_leaf": rf_min_samples_leaf,
            "rf_max_features": rf_max_features,
            "rf_bootstrap": rf_bootstrap,

            # ET base
            "et_n_estimators": et_n_estimators,
            "et_max_depth": et_max_depth,
            "et_min_samples_split": et_min_samples_split,
            "et_min_samples_leaf": et_min_samples_leaf,
            "et_max_features": et_max_features,
            "et_bootstrap": et_bootstrap,

            # meta-learner LR
            "meta_C": meta_C,
            "meta_max_iter": meta_max_iter
        }

        mean_recall = evaluate_joint_params_inner_cv(
            X_train_outer,
            y_train_outer,
            params_joint,
            n_splits_inner=n_splits_inner,
            random_state_inner=random_state_inner
        )

        # maximize recall
        return mean_recall

    study = optuna.create_study(
        direction="maximize",
        sampler=TPESampler(seed=seed_optuna),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    best_params_joint = study.best_params.copy()
    best_inner_recall = study.best_value  # Mean recall from inner CV.

    return best_params_joint, best_inner_recall


# =========================================================
# MAIN: Outer nested CV and fold-level reporting.
# =========================================================
def nested_cv_pipeline_stacking_joint(
    X,
    y,
    outer_splits=5,
    inner_splits=3,
    n_trials=30,
    seed_outer=42,
    seed_inner=123,
    seed_optuna=42
):
    """
    outer_splits: Number of folds for final evaluation (e.g., 5).
    inner_splits: Number of folds for tuning (e.g., 3).
    n_trials:        Optuna  outer fold
    """

    outer_kf = StratifiedKFold(
        n_splits=outer_splits,
        shuffle=True,
        random_state=seed_outer
    )

    fold_metrics = []
    fold_params = []

    print("===== Nested CV Stacking (joint Bayesian tuning for RF + ET + meta-LR) =====\n")

    for fold_idx, (tr_idx, te_idx) in enumerate(outer_kf.split(X, y), start=1):
        X_train_outer = X.iloc[tr_idx] if hasattr(X, "iloc") else X[tr_idx]
        X_test_outer  = X.iloc[te_idx] if hasattr(X, "iloc") else X[te_idx]
        y_train_outer = y.iloc[tr_idx] if hasattr(y, "iloc") else y[tr_idx]
        y_test_outer  = y.iloc[te_idx] if hasattr(y, "iloc") else y[te_idx]

        # -------------------------------------------------
        # -------------------------------------------------
        best_params_fold, best_inner_recall = tune_joint_params_for_outer_fold(
            X_train_outer,
            y_train_outer,
            n_trials=n_trials,
            n_splits_inner=inner_splits,
            random_state_inner=seed_inner,
            seed_optuna=seed_optuna
        )

        fold_params.append(best_params_fold)

        # -------------------------------------------------
        # -------------------------------------------------
        smoteenn = SMOTEENN(random_state=RANDOM_STATE)
        X_train_res, y_train_res = smoteenn.fit_resample(X_train_outer, y_train_outer)

        model_final = build_stacking_from_params(best_params_fold)
        model_final.fit(X_train_res, y_train_res)

        y_pred = model_final.predict(X_test_outer)
        y_proba = model_final.predict_proba(X_test_outer)[:, 1]

        acc  = accuracy_score(y_test_outer, y_pred)
        prec = precision_score(y_test_outer, y_pred, zero_division=0)
        rec  = recall_score(y_test_outer, y_pred, zero_division=0)
        f1   = f1_score(y_test_outer, y_pred, zero_division=0)
        auc  = roc_auc_score(y_test_outer, y_proba)

        cm = confusion_matrix(y_test_outer, y_pred)

        fold_metrics.append({
            "fold": fold_idx,
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "f1": f1,
            "auc": auc,
            "TN": int(cm[0,0]),
            "FP": int(cm[0,1]),
            "FN": int(cm[1,0]),
            "TP": int(cm[1,1]),
            "best_inner_recall": best_inner_recall
        })

        # -------------------------------------------------
        # 3) Display fold-level results.
        # -------------------------------------------------
        print(f"[Outer Fold {fold_idx}]")
        print(f"  Inner-CV tuned recall (objective): {best_inner_recall:.4f}")
        print(f"  Test Performance on this fold:")
        print(f"    Acc={acc:.4f}  Prec={prec:.4f}  Rec={rec:.4f}  "
              f"F1={f1:.4f}  AUC={auc:.4f}")
        print("  Confusion Matrix:")
        print(cm)
        print("  Best Joint Params:")
        print(best_params_fold)
        print("-"*100)

    # -------------------------------------------------
    # 4) Summarize results across all outer folds.
    # -------------------------------------------------
    df_res = pd.DataFrame(fold_metrics)

    print("\n===== Summary over all outer folds =====")
    for m in ["accuracy", "precision", "recall", "f1", "auc"]:
        mean_val = df_res[m].mean()
        std_val  = df_res[m].std(ddof=1)
        print(f"{m.capitalize():<10} mean={mean_val:.4f}  std={std_val:.4f}")

    print("\nPer-fold confusion / best params:")
    for row in fold_metrics:
        print(f" Fold {row['fold']}: "
              f"TN={row['TN']} FP={row['FP']} FN={row['FN']} TP={row['TP']}")
        print(f"  Params: {fold_params[row['fold']-1]}")
        print()

    summary_results = {
        "Accuracy":  (df_res["accuracy"].mean(),  df_res["accuracy"].std(ddof=1)),
        "Precision": (df_res["precision"].mean(), df_res["precision"].std(ddof=1)),
        "Recall":    (df_res["recall"].mean(),    df_res["recall"].std(ddof=1)),
        "F1":        (df_res["f1"].mean(),        df_res["f1"].std(ddof=1)),
        "AUC":       (df_res["auc"].mean(),       df_res["auc"].std(ddof=1)),
    }

    return summary_results, fold_metrics, fold_params


# =========================================================
# Usage:
# X = df.drop("target", axis=1)
# y = df["target"]
#
summary_results, fold_metrics, params_each_fold = nested_cv_pipeline_stacking_joint(
    X, y,
    outer_splits=5,
    inner_splits=3,
    n_trials=30,
    seed_outer=42,
    seed_inner=123,
    seed_optuna=42
)
#
# fold_metrics     -> fold-level performance and confusion matrix. counts
# =========================================================


# Tuning RF+LR+ET Conventional using Bayesian optimization

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
from imblearn.combine import SMOTEENN

import optuna
from optuna.samplers import TPESampler


# =========================================================
#   # Logistic Regression base
#   lr_C, lr_max_iter
#
#   # RandomForest base
#   rf_n_estimators, rf_max_depth, rf_min_samples_split,
#   rf_min_samples_leaf, rf_max_features, rf_bootstrap
#
#   # ExtraTrees base
#   et_n_estimators, et_max_depth, et_min_samples_split,
#   et_min_samples_leaf, et_max_features, et_bootstrap
#
#   # Meta-learner (Logistic Regression)
#   meta_C, meta_max_iter
# =========================================================
def build_stacking_from_params(params_joint):
    # ----- base: Logistic Regression (+ scaling) -----
    lr_pipe = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=params_joint["lr_C"],
            max_iter=params_joint["lr_max_iter"],
            random_state=RANDOM_STATE
        )
    )

    # ----- base: RandomForest (+ scaling) -----
    rf_pipe = make_pipeline(
        StandardScaler(),
        RandomForestClassifier(
            n_estimators=params_joint["rf_n_estimators"],
            max_depth=params_joint["rf_max_depth"],
            min_samples_split=params_joint["rf_min_samples_split"],
            min_samples_leaf=params_joint["rf_min_samples_leaf"],
            max_features=params_joint["rf_max_features"],
            bootstrap=params_joint["rf_bootstrap"],
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    )

    # ----- base: ExtraTrees (+ scaling) -----
    et_pipe = make_pipeline(
        StandardScaler(),
        ExtraTreesClassifier(
            n_estimators=params_joint["et_n_estimators"],
            max_depth=params_joint["et_max_depth"],
            min_samples_split=params_joint["et_min_samples_split"],
            min_samples_leaf=params_joint["et_min_samples_leaf"],
            max_features=params_joint["et_max_features"],
            bootstrap=params_joint["et_bootstrap"],
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    )

    # ----- meta-learner: Logistic Regression -----
    final_lr = LogisticRegression(
        C=params_joint["meta_C"],
        max_iter=params_joint["meta_max_iter"],
        random_state=RANDOM_STATE
    )

    stack_model = StackingClassifier(
        estimators=[
            ('LR', lr_pipe),
            ('RF', rf_pipe),
            ('ET', et_pipe)
        ],
        final_estimator=final_lr,
        cv=5,
        passthrough=False,
        n_jobs=-1
    )

    return stack_model


# =========================================================
# =========================================================
def evaluate_joint_params_inner_cv(
    X_train_outer,
    y_train_outer,
    params_joint,
    n_splits_inner=3,
    random_state_inner=123
):
    """
     training set  outer fold
     inner CV ( 3-fold)
     inner split:
        - resample (SMOTEENN)  inner-train
        -  stacking  params_joint
        - predict inner-val
     mean recall  maximize  tuning
    """
    kf_inner = StratifiedKFold(
        n_splits=n_splits_inner,
        shuffle=True,
        random_state=random_state_inner
    )

    recalls = []

    for inner_tr_idx, inner_val_idx in kf_inner.split(X_train_outer, y_train_outer):
        X_tr, X_val = (
            X_train_outer.iloc[inner_tr_idx],
            X_train_outer.iloc[inner_val_idx],
        )
        y_tr, y_val = (
            y_train_outer.iloc[inner_tr_idx],
            y_train_outer.iloc[inner_val_idx],
        )

        # ===== Oversample only the inner-training set. =====
        smoteenn = SMOTEENN(random_state=42)
        X_tr_res, y_tr_res = smoteenn.fit_resample(X_tr, y_tr)

        # ===== train stacking params_joint =====
        model = build_stacking_from_params(params_joint)
        model.fit(X_tr_res, y_tr_res)

        # ===== Predict inner-val ( oversample) =====
        y_val_pred = model.predict(X_val)

        # ===== Metric: recall =====
        rec = recall_score(y_val, y_val_pred, zero_division=0)
        recalls.append(rec)

    return np.mean(recalls)


# =========================================================
# Return best_params_joint and best_inner_recall.
# =========================================================
def tune_joint_params_for_outer_fold(
    X_train_outer,
    y_train_outer,
    n_trials=30,
    n_splits_inner=3,
    random_state_inner=123,
    seed_optuna=42
):
    """
     Optuna :
    - Logistic Regression base
    - RandomForest base
    - ExtraTrees base
    - Meta Logistic Regression
     (joint)
     Objective: mean recall from inner CV.
    """

    def objective(trial):
        # -------- Base LR params --------
        lr_C = trial.suggest_float("lr_C", 1e-4, 1e2, log=True)
        lr_max_iter = trial.suggest_int("lr_max_iter", 100, 2000)

        # -------- Base RF params --------
        rf_n_estimators = trial.suggest_int("rf_n_estimators", 50, 300)
        rf_max_depth = trial.suggest_int("rf_max_depth", 3, 50)
        rf_min_samples_split = trial.suggest_int("rf_min_samples_split", 2, 20)
        rf_min_samples_leaf = trial.suggest_int("rf_min_samples_leaf", 1, 10)
        rf_max_features = trial.suggest_categorical(
            "rf_max_features", ["sqrt", "log2", None]
        )
        rf_bootstrap = trial.suggest_categorical(
            "rf_bootstrap", [True, False]
        )

        # -------- Base ExtraTrees params --------
        et_n_estimators = trial.suggest_int("et_n_estimators", 50, 300)
        et_max_depth = trial.suggest_int("et_max_depth", 3, 50)
        et_min_samples_split = trial.suggest_int("et_min_samples_split", 2, 20)
        et_min_samples_leaf = trial.suggest_int("et_min_samples_leaf", 1, 10)
        et_max_features = trial.suggest_categorical(
            "et_max_features", ["sqrt", "log2", None]
        )
        et_bootstrap = trial.suggest_categorical(
            "et_bootstrap", [True, False]
        )

        # -------- Meta LR params --------
        meta_C = trial.suggest_float("meta_C", 1e-4, 1e2, log=True)
        meta_max_iter = trial.suggest_int("meta_max_iter", 100, 2000)

        params_joint = {
            # LR base
            "lr_C": lr_C,
            "lr_max_iter": lr_max_iter,

            # RF base
            "rf_n_estimators": rf_n_estimators,
            "rf_max_depth": rf_max_depth,
            "rf_min_samples_split": rf_min_samples_split,
            "rf_min_samples_leaf": rf_min_samples_leaf,
            "rf_max_features": rf_max_features,
            "rf_bootstrap": rf_bootstrap,

            # ET base
            "et_n_estimators": et_n_estimators,
            "et_max_depth": et_max_depth,
            "et_min_samples_split": et_min_samples_split,
            "et_min_samples_leaf": et_min_samples_leaf,
            "et_max_features": et_max_features,
            "et_bootstrap": et_bootstrap,

            # Meta LR
            "meta_C": meta_C,
            "meta_max_iter": meta_max_iter
        }

        mean_recall = evaluate_joint_params_inner_cv(
            X_train_outer,
            y_train_outer,
            params_joint,
            n_splits_inner=n_splits_inner,
            random_state_inner=random_state_inner
        )

        # maximize recall
        return mean_recall

    study = optuna.create_study(
        direction="maximize",
        sampler=TPESampler(seed=seed_optuna),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    best_params_joint = study.best_params.copy()
    best_inner_recall = study.best_value  # Mean recall from inner CV.

    return best_params_joint, best_inner_recall


# =========================================================
# MAIN: Outer nested CV and fold-level reporting.
# =========================================================
def nested_cv_pipeline_stacking_joint(
    X,
    y,
    outer_splits=5,
    inner_splits=3,
    n_trials=30,
    seed_outer=42,
    seed_inner=123,
    seed_optuna=42
):
    """
    outer_splits: Number of folds for final evaluation (e.g., 5).
    inner_splits: Number of folds for tuning (e.g., 3).
    n_trials:        Optuna  outer fold
    """

    outer_kf = StratifiedKFold(
        n_splits=outer_splits,
        shuffle=True,
        random_state=seed_outer
    )

    fold_metrics = []
    fold_params = []

    print("===== Nested CV Stacking (joint Bayesian tuning for RF + LR + ET + meta-LR) =====\n")

    for fold_idx, (tr_idx, te_idx) in enumerate(outer_kf.split(X, y), start=1):
        # ---------------------------
        # ---------------------------
        X_train_outer = X.iloc[tr_idx] if hasattr(X, "iloc") else X[tr_idx]
        X_test_outer  = X.iloc[te_idx] if hasattr(X, "iloc") else X[te_idx]
        y_train_outer = y.iloc[tr_idx] if hasattr(y, "iloc") else y[tr_idx]
        y_test_outer  = y.iloc[te_idx] if hasattr(y, "iloc") else y[te_idx]

        # ---------------------------
        # ---------------------------
        best_params_fold, best_inner_recall = tune_joint_params_for_outer_fold(
            X_train_outer,
            y_train_outer,
            n_trials=n_trials,
            n_splits_inner=inner_splits,
            random_state_inner=seed_inner,
            seed_optuna=seed_optuna + fold_idx
        )

        fold_params.append(best_params_fold)

        # ---------------------------
        # ---------------------------
        smoteenn = SMOTEENN(random_state=RANDOM_STATE)
        X_train_res, y_train_res = smoteenn.fit_resample(X_train_outer, y_train_outer)

        model_final = build_stacking_from_params(best_params_fold)
        model_final.fit(X_train_res, y_train_res)

        y_pred = model_final.predict(X_test_outer)
        y_proba = model_final.predict_proba(X_test_outer)[:, 1]

        acc  = accuracy_score(y_test_outer, y_pred)
        prec = precision_score(y_test_outer, y_pred, zero_division=0)
        rec  = recall_score(y_test_outer, y_pred, zero_division=0)
        f1   = f1_score(y_test_outer, y_pred, zero_division=0)
        auc  = roc_auc_score(y_test_outer, y_proba)

        cm = confusion_matrix(y_test_outer, y_pred)

        fold_metrics.append({
            "fold": fold_idx,
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "f1": f1,
            "auc": auc,
            "TN": int(cm[0,0]),
            "FP": int(cm[0,1]),
            "FN": int(cm[1,0]),
            "TP": int(cm[1,1]),
            "best_inner_recall": best_inner_recall
        })

        # ---------------------------
        # 3) Display fold-level results.
        # ---------------------------
        print(f"[Outer Fold {fold_idx}]")
        print(f"  Inner-CV tuned recall (objective): {best_inner_recall:.4f}")
        print(f"  Test Performance on this fold:")
        print(f"    Acc={acc:.4f}  Prec={prec:.4f}  Rec={rec:.4f}  "
              f"F1={f1:.4f}  AUC={auc:.4f}")
        print("  Confusion Matrix:")
        print(cm)
        print("  Best Joint Params:")
        print(best_params_fold)
        print("-"*100)

    # ---------------------------
    # 4) Summarize results across all outer folds.
    # ---------------------------
    df_res = pd.DataFrame(fold_metrics)

    print("\n===== Summary over all outer folds =====")
    for m in ["accuracy", "precision", "recall", "f1", "auc"]:
        mean_val = df_res[m].mean()
        std_val  = df_res[m].std(ddof=1)
        print(f"{m.capitalize():<10} mean={mean_val:.4f}  std={std_val:.4f}")

    print("\nPer-fold confusion / best params:")
    for row in fold_metrics:
        print(f" Fold {row['fold']}: "
              f"TN={row['TN']} FP={row['FP']} FN={row['FN']} TP={row['TP']}")
        print(f"  Params: {fold_params[row['fold']-1]}")
        print()

    summary_results = {
        "Accuracy":  (df_res["accuracy"].mean(),  df_res["accuracy"].std(ddof=1)),
        "Precision": (df_res["precision"].mean(), df_res["precision"].std(ddof=1)),
        "Recall":    (df_res["recall"].mean(),    df_res["recall"].std(ddof=1)),
        "F1":        (df_res["f1"].mean(),        df_res["f1"].std(ddof=1)),
        "AUC":       (df_res["auc"].mean(),       df_res["auc"].std(ddof=1)),
    }

    return summary_results, fold_metrics, fold_params


# =========================================================
# Usage:
# X = df.drop("target", axis=1)
# y = df["target"]
#
# Then call:
#
summary_results, fold_metrics, params_each_fold = nested_cv_pipeline_stacking_joint(
      X, y,
      outer_splits=5,
      inner_splits=3,
      n_trials=100,
     seed_outer=42,
      seed_inner=123,
      seed_optuna=42
 )
#
# fold_metrics     -> fold-level performance and confusion matrix. counts
# =========================================================


# Tuning LR+SVM+ET Conventional using Bayesian optimization

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.ensemble import StackingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
from imblearn.combine import SMOTEENN

import optuna
from optuna.samplers import TPESampler


# =========================================================
#
#   lr_C, lr_max_iter
#
#   svm_C, svm_kernel, svm_gamma, svm_degree
#
#   et_n_estimators, et_max_depth, et_min_samples_split,
#   et_min_samples_leaf, et_max_features, et_bootstrap
#
#   meta_C, meta_max_iter
# =========================================================
def build_stacking_from_params(params_joint):
    # -------- base: Logistic Regression --------
    lr_pipe = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=params_joint["lr_C"],
            max_iter=params_joint["lr_max_iter"],
            random_state=RANDOM_STATE
        )
    )

    # -------- base: SVM --------
    svm_kwargs = {
        "C": params_joint["svm_C"],
        "kernel": params_joint["svm_kernel"],
        "probability": True,
        "random_state": 0
    }

    if params_joint["svm_kernel"] in ["rbf", "poly"]:
        svm_kwargs["gamma"] = params_joint["svm_gamma"]

    if params_joint["svm_kernel"] == "poly":
        svm_kwargs["degree"] = params_joint["svm_degree"]

    svm_pipe = make_pipeline(
        StandardScaler(),
        SVC(**svm_kwargs)
    )

    # -------- base: ExtraTrees --------
    et_pipe = make_pipeline(
        StandardScaler(),
        ExtraTreesClassifier(
            n_estimators=params_joint["et_n_estimators"],
            max_depth=params_joint["et_max_depth"],
            min_samples_split=params_joint["et_min_samples_split"],
            min_samples_leaf=params_joint["et_min_samples_leaf"],
            max_features=params_joint["et_max_features"],
            bootstrap=params_joint["et_bootstrap"],
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    )

    # -------- meta-learner: Logistic Regression --------
    final_lr = LogisticRegression(
        C=params_joint["meta_C"],
        max_iter=params_joint["meta_max_iter"],
        random_state=RANDOM_STATE
    )

    # -------- StackingClassifier --------
    stack_model = StackingClassifier(
        estimators=[
            ('LR',  lr_pipe),
            ('SVM', svm_pipe),
            ('ET',  et_pipe)
        ],
        final_estimator=final_lr,
        cv=5,
        passthrough=False,
        n_jobs=-1
    )

    return stack_model


# =========================================================
# =========================================================
def evaluate_joint_params_inner_cv(
    X_train_outer,
    y_train_outer,
    params_joint,
    n_splits_inner=3,
    random_state_inner=123
):
    """
     training set  outer fold
     inner CV ( 3-fold)
     inner split:
        - resample (SMOTEENN)  inner-train
        -  stacking  params_joint
        - predict inner-val
     mean recall  maximize  tuning
    """
    kf_inner = StratifiedKFold(
        n_splits=n_splits_inner,
        shuffle=True,
        random_state=random_state_inner
    )

    recalls = []

    for inner_tr_idx, inner_val_idx in kf_inner.split(X_train_outer, y_train_outer):
        X_tr, X_val = (
            X_train_outer.iloc[inner_tr_idx],
            X_train_outer.iloc[inner_val_idx],
        )
        y_tr, y_val = (
            y_train_outer.iloc[inner_tr_idx],
            y_train_outer.iloc[inner_val_idx],
        )

        # ===== Oversample only the inner-training set. =====
        smoteenn = SMOTEENN(random_state=42)
        X_tr_res, y_tr_res = smoteenn.fit_resample(X_tr, y_tr)

        # ===== train stacking params_joint =====
        model = build_stacking_from_params(params_joint)
        model.fit(X_tr_res, y_tr_res)

        # ===== Predict inner-val ( oversample) =====
        y_val_pred = model.predict(X_val)

        # ===== Metric: recall =====
        rec = recall_score(y_val, y_val_pred, zero_division=0)
        recalls.append(rec)

    return np.mean(recalls)


# =========================================================
# Return best_params_joint and best_inner_recall.
# =========================================================
def tune_joint_params_for_outer_fold(
    X_train_outer,
    y_train_outer,
    n_trials=30,
    n_splits_inner=3,
    random_state_inner=123,
    seed_optuna=42
):
    """
     Optuna :
    - Logistic Regression base
    - SVM base
    - ExtraTrees base
    - Meta Logistic Regression
     (joint tuning)
     Objective: mean recall from inner CV.
    """

    def objective(trial):
        # -------- Base LR params --------
        lr_C = trial.suggest_float("lr_C", 1e-4, 1e2, log=True)
        lr_max_iter = trial.suggest_int("lr_max_iter", 100, 2000)

        # -------- SVM params --------
        svm_C = trial.suggest_float("svm_C", 1e-3, 1e3, log=True)
        svm_kernel = trial.suggest_categorical(
            "svm_kernel", ["linear", "rbf", "poly"]
        )

        if svm_kernel in ["rbf", "poly"]:
            svm_gamma = trial.suggest_float("svm_gamma", 1e-4, 1e1, log=True)
        else:
            svm_gamma = None

        if svm_kernel == "poly":
            svm_degree = trial.suggest_int("svm_degree", 2, 5)
        else:
            svm_degree = None

        # -------- ExtraTrees params --------
        et_n_estimators = trial.suggest_int("et_n_estimators", 50, 300)
        et_max_depth = trial.suggest_int("et_max_depth", 3, 50)
        et_min_samples_split = trial.suggest_int("et_min_samples_split", 2, 20)
        et_min_samples_leaf = trial.suggest_int("et_min_samples_leaf", 1, 10)
        et_max_features = trial.suggest_categorical(
            "et_max_features", ["sqrt", "log2", None]
        )
        et_bootstrap = trial.suggest_categorical(
            "et_bootstrap", [True, False]
        )

        # -------- Meta LR params --------
        meta_C = trial.suggest_float("meta_C", 1e-4, 1e2, log=True)
        meta_max_iter = trial.suggest_int("meta_max_iter", 100, 2000)

        params_joint = {
            # LR base
            "lr_C": lr_C,
            "lr_max_iter": lr_max_iter,

            # SVM base
            "svm_C": svm_C,
            "svm_kernel": svm_kernel,
            "svm_gamma": svm_gamma,
            "svm_degree": svm_degree,

            # ET base
            "et_n_estimators": et_n_estimators,
            "et_max_depth": et_max_depth,
            "et_min_samples_split": et_min_samples_split,
            "et_min_samples_leaf": et_min_samples_leaf,
            "et_max_features": et_max_features,
            "et_bootstrap": et_bootstrap,

            # Meta LR
            "meta_C": meta_C,
            "meta_max_iter": meta_max_iter
        }

        mean_recall = evaluate_joint_params_inner_cv(
            X_train_outer,
            y_train_outer,
            params_joint,
            n_splits_inner=n_splits_inner,
            random_state_inner=random_state_inner
        )

        return mean_recall

    study = optuna.create_study(
        direction="maximize",
        sampler=TPESampler(seed=seed_optuna),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    best_params_joint = study.best_params.copy()

    if "svm_gamma" not in best_params_joint:
        best_params_joint["svm_gamma"] = None
    if "svm_degree" not in best_params_joint:
        best_params_joint["svm_degree"] = None

    best_inner_recall = study.best_value  # Mean recall from inner CV.

    return best_params_joint, best_inner_recall


# =========================================================
# MAIN: Outer nested CV and fold-level reporting.
# =========================================================
def nested_cv_pipeline_stacking_joint(
    X,
    y,
    outer_splits=5,
    inner_splits=3,
    n_trials=30,
    seed_outer=42,
    seed_inner=123,
    seed_optuna=42
):
    """
    outer_splits: Number of folds for final evaluation (e.g., 5).
    inner_splits: Number of folds for tuning (e.g., 3).
    n_trials: Number of Optuna trials per outer fold.
    """

    outer_kf = StratifiedKFold(
        n_splits=outer_splits,
        shuffle=True,
        random_state=seed_outer
    )

    fold_metrics = []
    fold_params = []

    print("===== Nested CV Stacking (joint Bayesian tuning for LR + SVM + ET + meta-LR) =====\n")

    for fold_idx, (tr_idx, te_idx) in enumerate(outer_kf.split(X, y), start=1):
        X_train_outer = X.iloc[tr_idx] if hasattr(X, "iloc") else X[tr_idx]
        X_test_outer  = X.iloc[te_idx] if hasattr(X, "iloc") else X[te_idx]
        y_train_outer = y.iloc[tr_idx] if hasattr(y, "iloc") else y[tr_idx]
        y_test_outer  = y.iloc[te_idx] if hasattr(y, "iloc") else y[te_idx]

        # -------------------------------------------------
        # -------------------------------------------------
        best_params_fold, best_inner_recall = tune_joint_params_for_outer_fold(
            X_train_outer,
            y_train_outer,
            n_trials=n_trials,
            n_splits_inner=inner_splits,
            random_state_inner=seed_inner,
            seed_optuna=seed_optuna + fold_idx
        )

        fold_params.append(best_params_fold)

        # -------------------------------------------------
        # -------------------------------------------------
        smoteenn = SMOTEENN(random_state=RANDOM_STATE)
        X_train_res, y_train_res = smoteenn.fit_resample(X_train_outer, y_train_outer)

        model_final = build_stacking_from_params(best_params_fold)
        model_final.fit(X_train_res, y_train_res)

        y_pred = model_final.predict(X_test_outer)
        y_proba = model_final.predict_proba(X_test_outer)[:, 1]

        acc  = accuracy_score(y_test_outer, y_pred)
        prec = precision_score(y_test_outer, y_pred, zero_division=0)
        rec  = recall_score(y_test_outer, y_pred, zero_division=0)
        f1   = f1_score(y_test_outer, y_pred, zero_division=0)
        auc  = roc_auc_score(y_test_outer, y_proba)

        cm = confusion_matrix(y_test_outer, y_pred)

        fold_metrics.append({
            "fold": fold_idx,
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "f1": f1,
            "auc": auc,
            "TN": int(cm[0,0]),
            "FP": int(cm[0,1]),
            "FN": int(cm[1,0]),
            "TP": int(cm[1,1]),
            "best_inner_recall": best_inner_recall
        })

        # -------------------------------------------------
        # 3) Display fold-level results.
        # -------------------------------------------------
        print(f"[Outer Fold {fold_idx}]")
        print(f"  Inner-CV tuned recall (objective): {best_inner_recall:.4f}")
        print(f"  Test Performance on this fold:")
        print(f"    Acc={acc:.4f}  Prec={prec:.4f}  Rec={rec:.4f}  "
              f"F1={f1:.4f}  AUC={auc:.4f}")
        print("  Confusion Matrix:")
        print(cm)
        print("  Best Joint Params:")
        print(best_params_fold)
        print("-"*100)

    # -------------------------------------------------
    # 4) Summarize results across all outer folds.
    # -------------------------------------------------
    df_res = pd.DataFrame(fold_metrics)

    print("\n===== Summary over all outer folds =====")
    for m in ["accuracy", "precision", "recall", "f1", "auc"]:
        mean_val = df_res[m].mean()
        std_val  = df_res[m].std(ddof=1)
        print(f"{m.capitalize():<10} mean={mean_val:.4f}  std={std_val:.4f}")

    print("\nPer-fold confusion / best params:")
    for row in fold_metrics:
        print(f" Fold {row['fold']}: "
              f"TN={row['TN']} FP={row['FP']} FN={row['FN']} TP={row['TP']}")
        print(f"  Params: {fold_params[row['fold']-1]}")
        print()

    summary_results = {
        "Accuracy":  (df_res["accuracy"].mean(),  df_res["accuracy"].std(ddof=1)),
        "Precision": (df_res["precision"].mean(), df_res["precision"].std(ddof=1)),
        "Recall":    (df_res["recall"].mean(),    df_res["recall"].std(ddof=1)),
        "F1":        (df_res["f1"].mean(),        df_res["f1"].std(ddof=1)),
        "AUC":       (df_res["auc"].mean(),       df_res["auc"].std(ddof=1)),
    }

    return summary_results, fold_metrics, fold_params


# =========================================================
# Usage:
# X = df.drop("target", axis=1)
# y = df["target"]
#
# Then call:
#
summary_results, fold_metrics, params_each_fold = nested_cv_pipeline_stacking_joint(
      X, y,
      outer_splits=5,
      inner_splits=3,
      n_trials=100,
      seed_outer=42,
      seed_inner=123,
      seed_optuna=42
  )
#
# fold_metrics     -> fold-level performance and confusion matrix. counts
# =========================================================


# Tuning RF+SVM+ET Conventional using Bayesian optimization

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.ensemble import StackingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
from imblearn.combine import SMOTEENN

import optuna
from optuna.samplers import TPESampler


# =========================================================
#
#   rf_n_estimators, rf_max_depth, rf_min_samples_split,
#   rf_min_samples_leaf, rf_max_features, rf_bootstrap
#
#   svm_C, svm_kernel, svm_gamma, svm_degree
#
#   et_n_estimators, et_max_depth, et_min_samples_split,
#   et_min_samples_leaf, et_max_features, et_bootstrap
#
#   meta_C, meta_max_iter
# =========================================================
def build_stacking_from_params(params_joint):
    # -------- base: RandomForest --------
    rf_pipe = make_pipeline(
        StandardScaler(),
        RandomForestClassifier(
            n_estimators=params_joint["rf_n_estimators"],
            max_depth=params_joint["rf_max_depth"],
            min_samples_split=params_joint["rf_min_samples_split"],
            min_samples_leaf=params_joint["rf_min_samples_leaf"],
            max_features=params_joint["rf_max_features"],
            bootstrap=params_joint["rf_bootstrap"],
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    )

    # -------- base: SVM --------
    svm_kwargs = {
        "C": params_joint["svm_C"],
        "kernel": params_joint["svm_kernel"],
        "probability": True,
        "random_state": 0
    }

    if params_joint["svm_kernel"] in ["rbf", "poly"]:
        svm_kwargs["gamma"] = params_joint["svm_gamma"]

    if params_joint["svm_kernel"] == "poly":
        svm_kwargs["degree"] = params_joint["svm_degree"]

    svm_pipe = make_pipeline(
        StandardScaler(),
        SVC(**svm_kwargs)
    )

    # -------- base: ExtraTrees --------
    et_pipe = make_pipeline(
        StandardScaler(),
        ExtraTreesClassifier(
            n_estimators=params_joint["et_n_estimators"],
            max_depth=params_joint["et_max_depth"],
            min_samples_split=params_joint["et_min_samples_split"],
            min_samples_leaf=params_joint["et_min_samples_leaf"],
            max_features=params_joint["et_max_features"],
            bootstrap=params_joint["et_bootstrap"],
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    )

    # -------- meta-learner: Logistic Regression --------
    final_lr = LogisticRegression(
        C=params_joint["meta_C"],
        max_iter=params_joint["meta_max_iter"],
        random_state=RANDOM_STATE
    )

    # -------- StackingClassifier --------
    stack_model = StackingClassifier(
        estimators=[
            ('RF',  rf_pipe),
            ('SVM', svm_pipe),
            ('ET',  et_pipe)
        ],
        final_estimator=final_lr,
        cv=5,
        passthrough=False,
        n_jobs=-1
    )

    return stack_model


# =========================================================
# =========================================================
def evaluate_joint_params_inner_cv(
    X_train_outer,
    y_train_outer,
    params_joint,
    n_splits_inner=3,
    random_state_inner=123
):
    """
     training set  outer fold
     inner CV ( 3-fold)
     inner split:
        - resample (SMOTEENN)  inner-train
        -  stacking  params_joint
        - predict inner-val
     mean recall  maximize  tuning
    """
    kf_inner = StratifiedKFold(
        n_splits=n_splits_inner,
        shuffle=True,
        random_state=random_state_inner
    )

    recalls = []

    for inner_tr_idx, inner_val_idx in kf_inner.split(X_train_outer, y_train_outer):
        X_tr, X_val = (
            X_train_outer.iloc[inner_tr_idx],
            X_train_outer.iloc[inner_val_idx],
        )
        y_tr, y_val = (
            y_train_outer.iloc[inner_tr_idx],
            y_train_outer.iloc[inner_val_idx],
        )

        # ===== Oversample only the inner-training set. =====
        smoteenn = SMOTEENN(random_state=42)
        X_tr_res, y_tr_res = smoteenn.fit_resample(X_tr, y_tr)

        # ===== train stacking params_joint =====
        model = build_stacking_from_params(params_joint)
        model.fit(X_tr_res, y_tr_res)

        # ===== Predict inner-val ( oversample) =====
        y_val_pred = model.predict(X_val)

        # ===== Metric: recall =====
        rec = recall_score(y_val, y_val_pred, zero_division=0)
        recalls.append(rec)

    return np.mean(recalls)


# =========================================================
# Return best_params_joint and best_inner_recall.
# =========================================================
def tune_joint_params_for_outer_fold(
    X_train_outer,
    y_train_outer,
    n_trials=30,
    n_splits_inner=3,
    random_state_inner=123,
    seed_optuna=42
):
    """
     Optuna :
    - RandomForest base
    - SVM base
    - ExtraTrees base
    - Meta Logistic Regression
     (joint tuning)
     Objective: mean recall from inner CV.
    """

    def objective(trial):
        # -------- RandomForest params --------
        rf_n_estimators = trial.suggest_int("rf_n_estimators", 50, 300)
        rf_max_depth = trial.suggest_int("rf_max_depth", 3, 50)
        rf_min_samples_split = trial.suggest_int("rf_min_samples_split", 2, 20)
        rf_min_samples_leaf = trial.suggest_int("rf_min_samples_leaf", 1, 10)
        rf_max_features = trial.suggest_categorical(
            "rf_max_features", ["sqrt", "log2", None]
        )
        rf_bootstrap = trial.suggest_categorical(
            "rf_bootstrap", [True, False]
        )

        # -------- SVM params --------
        svm_C = trial.suggest_float("svm_C", 1e-3, 1e3, log=True)
        svm_kernel = trial.suggest_categorical(
            "svm_kernel", ["linear", "rbf", "poly"]
        )

        if svm_kernel in ["rbf", "poly"]:
            svm_gamma = trial.suggest_float("svm_gamma", 1e-4, 1e1, log=True)
        else:
            svm_gamma = None

        if svm_kernel == "poly":
            svm_degree = trial.suggest_int("svm_degree", 2, 5)
        else:
            svm_degree = None

        # -------- ExtraTrees params --------
        et_n_estimators = trial.suggest_int("et_n_estimators", 50, 300)
        et_max_depth = trial.suggest_int("et_max_depth", 3, 50)
        et_min_samples_split = trial.suggest_int("et_min_samples_split", 2, 20)
        et_min_samples_leaf = trial.suggest_int("et_min_samples_leaf", 1, 10)
        et_max_features = trial.suggest_categorical(
            "et_max_features", ["sqrt", "log2", None]
        )
        et_bootstrap = trial.suggest_categorical(
            "et_bootstrap", [True, False]
        )

        # -------- Meta LR params --------
        meta_C = trial.suggest_float("meta_C", 1e-4, 1e2, log=True)
        meta_max_iter = trial.suggest_int("meta_max_iter", 100, 2000)

        params_joint = {
            # RF base
            "rf_n_estimators": rf_n_estimators,
            "rf_max_depth": rf_max_depth,
            "rf_min_samples_split": rf_min_samples_split,
            "rf_min_samples_leaf": rf_min_samples_leaf,
            "rf_max_features": rf_max_features,
            "rf_bootstrap": rf_bootstrap,

            # SVM base
            "svm_C": svm_C,
            "svm_kernel": svm_kernel,
            "svm_gamma": svm_gamma,
            "svm_degree": svm_degree,

            # ET base
            "et_n_estimators": et_n_estimators,
            "et_max_depth": et_max_depth,
            "et_min_samples_split": et_min_samples_split,
            "et_min_samples_leaf": et_min_samples_leaf,
            "et_max_features": et_max_features,
            "et_bootstrap": et_bootstrap,

            # Meta LR
            "meta_C": meta_C,
            "meta_max_iter": meta_max_iter
        }

        mean_recall = evaluate_joint_params_inner_cv(
            X_train_outer,
            y_train_outer,
            params_joint,
            n_splits_inner=n_splits_inner,
            random_state_inner=random_state_inner
        )

        # maximize recall
        return mean_recall

    study = optuna.create_study(
        direction="maximize",
        sampler=TPESampler(seed=seed_optuna),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    best_params_joint = study.best_params.copy()

    if "svm_gamma" not in best_params_joint:
        best_params_joint["svm_gamma"] = None
    if "svm_degree" not in best_params_joint:
        best_params_joint["svm_degree"] = None

    best_inner_recall = study.best_value  # Mean recall from inner CV.

    return best_params_joint, best_inner_recall


# =========================================================
# MAIN: Outer nested CV and fold-level reporting.
# =========================================================
def nested_cv_pipeline_stacking_joint(
    X,
    y,
    outer_splits=5,
    inner_splits=3,
    n_trials=30,
    seed_outer=42,
    seed_inner=123,
    seed_optuna=42
):
    """
    outer_splits: Number of folds for final evaluation (e.g., 5).
    inner_splits: Number of folds for tuning (e.g., 3).
    n_trials:        Optuna  outer fold
    """

    outer_kf = StratifiedKFold(
        n_splits=outer_splits,
        shuffle=True,
        random_state=seed_outer
    )

    fold_metrics = []
    fold_params = []

    print("===== Nested CV Stacking (joint Bayesian tuning for RF + SVM + ET + meta-LR) =====\n")

    for fold_idx, (tr_idx, te_idx) in enumerate(outer_kf.split(X, y), start=1):
        X_train_outer = X.iloc[tr_idx] if hasattr(X, "iloc") else X[tr_idx]
        X_test_outer  = X.iloc[te_idx] if hasattr(X, "iloc") else X[te_idx]
        y_train_outer = y.iloc[tr_idx] if hasattr(y, "iloc") else y[tr_idx]
        y_test_outer  = y.iloc[te_idx] if hasattr(y, "iloc") else y[te_idx]

        # -------------------------------------------------
        # -------------------------------------------------
        best_params_fold, best_inner_recall = tune_joint_params_for_outer_fold(
            X_train_outer,
            y_train_outer,
            n_trials=n_trials,
            n_splits_inner=inner_splits,
            random_state_inner=seed_inner,
            seed_optuna=seed_optuna + fold_idx
        )

        fold_params.append(best_params_fold)

        # -------------------------------------------------
        # -------------------------------------------------
        smoteenn = SMOTEENN(random_state=RANDOM_STATE)
        X_train_res, y_train_res = smoteenn.fit_resample(X_train_outer, y_train_outer)

        model_final = build_stacking_from_params(best_params_fold)
        model_final.fit(X_train_res, y_train_res)

        y_pred = model_final.predict(X_test_outer)
        y_proba = model_final.predict_proba(X_test_outer)[:, 1]

        acc  = accuracy_score(y_test_outer, y_pred)
        prec = precision_score(y_test_outer, y_pred, zero_division=0)
        rec  = recall_score(y_test_outer, y_pred, zero_division=0)
        f1   = f1_score(y_test_outer, y_pred, zero_division=0)
        auc  = roc_auc_score(y_test_outer, y_proba)

        cm = confusion_matrix(y_test_outer, y_pred)

        fold_metrics.append({
            "fold": fold_idx,
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "f1": f1,
            "auc": auc,
            "TN": int(cm[0,0]),
            "FP": int(cm[0,1]),
            "FN": int(cm[1,0]),
            "TP": int(cm[1,1]),
            "best_inner_recall": best_inner_recall
        })

        # -------------------------------------------------
        # 3) Display fold-level results.
        # -------------------------------------------------
        print(f"[Outer Fold {fold_idx}]")
        print(f"  Inner-CV tuned recall (objective): {best_inner_recall:.4f}")
        print(f"  Test Performance on this fold:")
        print(f"    Acc={acc:.4f}  Prec={prec:.4f}  Rec={rec:.4f}  "
              f"F1={f1:.4f}  AUC={auc:.4f}")
        print("  Confusion Matrix:")
        print(cm)
        print("  Best Joint Params:")
        print(best_params_fold)
        print("-"*100)

    # -------------------------------------------------
    # 4) Summarize results across all outer folds.
    # -------------------------------------------------
    df_res = pd.DataFrame(fold_metrics)

    print("\n===== Summary over all outer folds =====")
    for m in ["accuracy", "precision", "recall", "f1", "auc"]:
        mean_val = df_res[m].mean()
        std_val  = df_res[m].std(ddof=1)
        print(f"{m.capitalize():<10} mean={mean_val:.4f}  std={std_val:.4f}")

    print("\nPer-fold confusion / best params:")
    for row in fold_metrics:
        print(f" Fold {row['fold']}: "
              f"TN={row['TN']} FP={row['FP']} FN={row['FN']} TP={row['TP']}")
        print(f"  Params: {fold_params[row['fold']-1]}")
        print()

    summary_results = {
        "Accuracy":  (df_res["accuracy"].mean(),  df_res["accuracy"].std(ddof=1)),
        "Precision": (df_res["precision"].mean(), df_res["precision"].std(ddof=1)),
        "Recall":    (df_res["recall"].mean(),    df_res["recall"].std(ddof=1)),
        "F1":        (df_res["f1"].mean(),        df_res["f1"].std(ddof=1)),
        "AUC":       (df_res["auc"].mean(),       df_res["auc"].std(ddof=1)),
    }

    return summary_results, fold_metrics, fold_params


# =========================================================
# Usage:
# X = df.drop("target", axis=1)
# y = df["target"]
#
# Then call:
#
summary_results, fold_metrics, params_each_fold = nested_cv_pipeline_stacking_joint(
      X, y,
      outer_splits=5,
      inner_splits=3,
      n_trials=100,
      seed_outer=42,
      seed_inner=123,
      seed_optuna=42
  )
#
# fold_metrics     -> fold-level performance and confusion matrix. counts
# =========================================================


# Tuning SVM+ET Conventional using Bayesian optimization

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
from imblearn.combine import SMOTEENN

import optuna
from optuna.samplers import TPESampler


# =========================================================
#   # SVM base
#   svm_C, svm_kernel, svm_gamma
#
#   # ExtraTrees base
#   et_n_estimators, et_max_depth,
#   et_min_samples_split, et_min_samples_leaf,
#   et_max_features, et_bootstrap
#
#   # Meta-learner (Logistic Regression)
#   meta_C, meta_max_iter
# =========================================================
def build_stacking_from_params(params_joint):
    # ----- base: SVM (+ scaling) -----
    svm_pipe = make_pipeline(
        StandardScaler(),
        SVC(
            C=params_joint["svm_C"],
            kernel=params_joint["svm_kernel"],
            gamma=params_joint["svm_gamma"],
            probability=True,
            random_state=RANDOM_STATE
        )
    )

    # ----- base: ExtraTrees (+ scaling) -----
    et_pipe = make_pipeline(
        StandardScaler(),
        ExtraTreesClassifier(
            n_estimators=params_joint["et_n_estimators"],
            max_depth=params_joint["et_max_depth"],
            min_samples_split=params_joint["et_min_samples_split"],
            min_samples_leaf=params_joint["et_min_samples_leaf"],
            max_features=params_joint["et_max_features"],
            bootstrap=params_joint["et_bootstrap"],
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    )

    # ----- meta-learner: Logistic Regression -----
    final_lr = LogisticRegression(
        C=params_joint["meta_C"],
        max_iter=params_joint["meta_max_iter"],
        random_state=RANDOM_STATE
    )

    stack_model = StackingClassifier(
        estimators=[
            ('SVM', svm_pipe),
            ('ET',  et_pipe)
        ],
        final_estimator=final_lr,
        cv=5,
        passthrough=False,
        n_jobs=-1
    )

    return stack_model


# =========================================================
# =========================================================
def evaluate_joint_params_inner_cv(
    X_train_outer,
    y_train_outer,
    params_joint,
    n_splits_inner=3,
    random_state_inner=123
):
    """
     training set  outer fold
     inner CV ( 3-fold)
     inner split:
        - resample (SMOTEENN)  inner-train
        -  stacking  params_joint
        - predict inner-val
     mean recall  maximize  tuning
    """
    kf_inner = StratifiedKFold(
        n_splits=n_splits_inner,
        shuffle=True,
        random_state=random_state_inner
    )

    recalls = []

    for inner_tr_idx, inner_val_idx in kf_inner.split(X_train_outer, y_train_outer):
        X_tr, X_val = (
            X_train_outer.iloc[inner_tr_idx],
            X_train_outer.iloc[inner_val_idx],
        )
        y_tr, y_val = (
            y_train_outer.iloc[inner_tr_idx],
            y_train_outer.iloc[inner_val_idx],
        )

        # ===== Oversample only the inner-training set. =====
        smoteenn = SMOTEENN(random_state=42)
        X_tr_res, y_tr_res = smoteenn.fit_resample(X_tr, y_tr)

        # ===== train stacking params_joint =====
        model = build_stacking_from_params(params_joint)
        model.fit(X_tr_res, y_tr_res)

        # ===== Predict inner-val ( oversample) =====
        y_val_pred = model.predict(X_val)

        # ===== Metric: recall =====
        rec = recall_score(y_val, y_val_pred, zero_division=0)
        recalls.append(rec)

    return np.mean(recalls)


# =========================================================
# Return best_params_joint and best_inner_recall.
# =========================================================
def tune_joint_params_for_outer_fold(
    X_train_outer,
    y_train_outer,
    n_trials=30,
    n_splits_inner=3,
    random_state_inner=123,
    seed_optuna=42
):
    """
     Optuna :
    - SVM base
    - ExtraTrees base
    - Meta Logistic Regression
     (joint)
     Objective: mean recall from inner CV.
    """

    def objective(trial):
        # -------- Base SVM params --------
        svm_C = trial.suggest_float("svm_C", 1e-4, 1e2, log=True)
        svm_kernel = trial.suggest_categorical("svm_kernel", ["linear", "rbf"])
        svm_gamma = trial.suggest_categorical("svm_gamma", ["scale", "auto"])

        # -------- ExtraTrees params --------
        et_n_estimators = trial.suggest_int("et_n_estimators", 50, 300)
        et_max_depth = trial.suggest_int("et_max_depth", 3, 50)
        et_min_samples_split = trial.suggest_int("et_min_samples_split", 2, 20)
        et_min_samples_leaf = trial.suggest_int("et_min_samples_leaf", 1, 10)
        et_max_features = trial.suggest_categorical(
            "et_max_features", ["sqrt", "log2", None]
        )
        et_bootstrap = trial.suggest_categorical(
            "et_bootstrap", [True, False]
        )

        # -------- Meta LR params --------
        meta_C = trial.suggest_float("meta_C", 1e-4, 1e2, log=True)
        meta_max_iter = trial.suggest_int("meta_max_iter", 100, 2000)

        params_joint = {
            "svm_C": svm_C,
            "svm_kernel": svm_kernel,
            "svm_gamma": svm_gamma,

            "et_n_estimators": et_n_estimators,
            "et_max_depth": et_max_depth,
            "et_min_samples_split": et_min_samples_split,
            "et_min_samples_leaf": et_min_samples_leaf,
            "et_max_features": et_max_features,
            "et_bootstrap": et_bootstrap,

            "meta_C": meta_C,
            "meta_max_iter": meta_max_iter
        }

        mean_recall = evaluate_joint_params_inner_cv(
            X_train_outer,
            y_train_outer,
            params_joint,
            n_splits_inner=n_splits_inner,
            random_state_inner=random_state_inner
        )

        # maximize recall
        return mean_recall

    study = optuna.create_study(
        direction="maximize",
        sampler=TPESampler(seed=seed_optuna),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    best_params_joint = study.best_params.copy()
    best_inner_recall = study.best_value  # Mean recall from inner CV.

    return best_params_joint, best_inner_recall


# =========================================================
# MAIN: Outer nested CV and fold-level reporting.
# =========================================================
def nested_cv_pipeline_stacking_joint(
    X,
    y,
    outer_splits=5,
    inner_splits=3,
    n_trials=30,
    seed_outer=42,
    seed_inner=123,
    seed_optuna=42
):
    """
    outer_splits: Number of folds for final evaluation (e.g., 5).
    inner_splits: Number of folds for tuning (e.g., 3).
    n_trials:        Optuna  outer fold
    """

    outer_kf = StratifiedKFold(
        n_splits=outer_splits,
        shuffle=True,
        random_state=seed_outer
    )

    fold_metrics = []
    fold_params = []

    print("===== Nested CV Stacking (joint Bayesian tuning for SVM + ET + meta-LR) =====\n")

    for fold_idx, (tr_idx, te_idx) in enumerate(outer_kf.split(X, y), start=1):
        X_train_outer = X.iloc[tr_idx] if hasattr(X, "iloc") else X[tr_idx]
        X_test_outer  = X.iloc[te_idx] if hasattr(X, "iloc") else X[te_idx]
        y_train_outer = y.iloc[tr_idx] if hasattr(y, "iloc") else y[tr_idx]
        y_test_outer  = y.iloc[te_idx] if hasattr(y, "iloc") else y[te_idx]

        # -------------------------------------------------
        # -------------------------------------------------
        best_params_fold, best_inner_recall = tune_joint_params_for_outer_fold(
            X_train_outer,
            y_train_outer,
            n_trials=n_trials,
            n_splits_inner=inner_splits,
            random_state_inner=seed_inner,
            seed_optuna=seed_optuna + fold_idx
        )

        fold_params.append(best_params_fold)

        # -------------------------------------------------
        # -------------------------------------------------
        smoteenn = SMOTEENN(random_state=RANDOM_STATE)
        X_train_res, y_train_res = smoteenn.fit_resample(X_train_outer, y_train_outer)

        model_final = build_stacking_from_params(best_params_fold)
        model_final.fit(X_train_res, y_train_res)

        y_pred = model_final.predict(X_test_outer)
        y_proba = model_final.predict_proba(X_test_outer)[:, 1]

        acc  = accuracy_score(y_test_outer, y_pred)
        prec = precision_score(y_test_outer, y_pred, zero_division=0)
        rec  = recall_score(y_test_outer, y_pred, zero_division=0)
        f1   = f1_score(y_test_outer, y_pred, zero_division=0)
        auc  = roc_auc_score(y_test_outer, y_proba)

        cm = confusion_matrix(y_test_outer, y_pred)

        fold_metrics.append({
            "fold": fold_idx,
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "f1": f1,
            "auc": auc,
            "TN": int(cm[0,0]),
            "FP": int(cm[0,1]),
            "FN": int(cm[1,0]),
            "TP": int(cm[1,1]),
            "best_inner_recall": best_inner_recall
        })

        # -------------------------------------------------
        # 3) Display fold-level results.
        # -------------------------------------------------
        print(f"[Outer Fold {fold_idx}]")
        print(f"  Inner-CV tuned recall (objective): {best_inner_recall:.4f}")
        print(f"  Test Performance on this fold:")
        print(f"    Acc={acc:.4f}  Prec={prec:.4f}  Rec={rec:.4f}  "
              f"F1={f1:.4f}  AUC={auc:.4f}")
        print("  Confusion Matrix:")
        print(cm)
        print("  Best Joint Params:")
        print(best_params_fold)
        print("-"*100)

    # -------------------------------------------------
    # 4) Summarize results across all outer folds.
    # -------------------------------------------------
    df_res = pd.DataFrame(fold_metrics)

    print("\n===== Summary over all outer folds =====")
    for m in ["accuracy", "precision", "recall", "f1", "auc"]:
        mean_val = df_res[m].mean()
        std_val  = df_res[m].std(ddof=1)
        print(f"{m.capitalize():<10} mean={mean_val:.4f}  std={std_val:.4f}")

    print("\nPer-fold confusion / best params:")
    for row in fold_metrics:
        print(f" Fold {row['fold']}: "
              f"TN={row['TN']} FP={row['FP']} FN={row['FN']} TP={row['TP']}")
        print(f"  Params: {fold_params[row['fold']-1]}")
        print()

    summary_results = {
        "Accuracy":  (df_res["accuracy"].mean(),  df_res["accuracy"].std(ddof=1)),
        "Precision": (df_res["precision"].mean(), df_res["precision"].std(ddof=1)),
        "Recall":    (df_res["recall"].mean(),    df_res["recall"].std(ddof=1)),
        "F1":        (df_res["f1"].mean(),        df_res["f1"].std(ddof=1)),
        "AUC":       (df_res["auc"].mean(),       df_res["auc"].std(ddof=1)),
    }

    return summary_results, fold_metrics, fold_params


# =========================================================
# Usage:
# X = df.drop("target", axis=1)
# y = df["target"]
#
# Then call:
#
# summary_results, fold_metrics, params_each_fold = nested_cv_pipeline_stacking_joint(
#      X, y,
#      outer_splits=5,
#      inner_splits=3,
#      n_trials=100,
#      seed_outer=42,
#      seed_inner=123,
#      seed_optuna=42
#  )
#
# fold_metrics     -> fold-level performance and confusion matrix. counts
# =========================================================


# Tuning RF+LR+SVM+ET Conventional k-fold hyperparameter tuning using Bayesian optimization

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.ensemble import StackingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
from imblearn.combine import SMOTEENN

import optuna
from optuna.samplers import TPESampler


# =========================================================
#
#
#   rf_n_estimators, rf_max_depth, rf_min_samples_split,
#   rf_min_samples_leaf, rf_max_features, rf_bootstrap
#
#   lr_C, lr_max_iter
#
#   svm_C, svm_kernel, svm_gamma, svm_degree
#       - svm_kernel ∈ {"linear","rbf","poly"}
#
#   et_n_estimators, et_max_depth, et_min_samples_split,
#   et_min_samples_leaf, et_max_features, et_bootstrap
#
#   meta_C, meta_max_iter
# =========================================================
def build_stacking_from_params(params_joint):
    # -------- base: RandomForest --------
    rf_pipe = make_pipeline(
        StandardScaler(),
        RandomForestClassifier(
            n_estimators=params_joint["rf_n_estimators"],
            max_depth=params_joint["rf_max_depth"],
            min_samples_split=params_joint["rf_min_samples_split"],
            min_samples_leaf=params_joint["rf_min_samples_leaf"],
            max_features=params_joint["rf_max_features"],
            bootstrap=params_joint["rf_bootstrap"],
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    )

    # -------- base: Logistic Regression --------
    lr_pipe = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=params_joint["lr_C"],
            max_iter=params_joint["lr_max_iter"],
            random_state=RANDOM_STATE
        )
    )

    # -------- base: SVM --------
    svm_kwargs = {
        "C": params_joint["svm_C"],
        "kernel": params_joint["svm_kernel"],
        "probability": True,
        "random_state": 0
    }

    if params_joint["svm_kernel"] in ["rbf", "poly"]:
        svm_kwargs["gamma"] = params_joint["svm_gamma"]

    if params_joint["svm_kernel"] == "poly":
        svm_kwargs["degree"] = params_joint["svm_degree"]

    svm_pipe = make_pipeline(
        StandardScaler(),
        SVC(**svm_kwargs)
    )

    # -------- base: ExtraTrees --------
    et_pipe = make_pipeline(
        StandardScaler(),
        ExtraTreesClassifier(
            n_estimators=params_joint["et_n_estimators"],
            max_depth=params_joint["et_max_depth"],
            min_samples_split=params_joint["et_min_samples_split"],
            min_samples_leaf=params_joint["et_min_samples_leaf"],
            max_features=params_joint["et_max_features"],
            bootstrap=params_joint["et_bootstrap"],
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    )

    # -------- meta-learner: Logistic Regression --------
    final_lr = LogisticRegression(
        C=params_joint["meta_C"],
        max_iter=params_joint["meta_max_iter"],
        random_state=RANDOM_STATE
    )

    # -------- StackingClassifier --------
    stack_model = StackingClassifier(
        estimators=[
            ('RF',  rf_pipe),
            ('LR',  lr_pipe),
            ('SVM', svm_pipe),
            ('ET',  et_pipe)
        ],
        final_estimator=final_lr,
        cv=5,
        passthrough=False,
        n_jobs=-1
    )

    return stack_model


# =========================================================
# =========================================================
def evaluate_joint_params_inner_cv(
    X_train_outer,
    y_train_outer,
    params_joint,
    n_splits_inner=3,
    random_state_inner=123
):
    """
     training set  outer fold
     inner CV ( 3-fold)
     inner split:
        - resample (SMOTEENN)  inner-train
        -  stacking  params_joint
        - predict inner-val
     mean recall  maximize  tuning
    """
    kf_inner = StratifiedKFold(
        n_splits=n_splits_inner,
        shuffle=True,
        random_state=random_state_inner
    )

    recalls = []

    for inner_tr_idx, inner_val_idx in kf_inner.split(X_train_outer, y_train_outer):
        X_tr, X_val = (
            X_train_outer.iloc[inner_tr_idx],
            X_train_outer.iloc[inner_val_idx],
        )
        y_tr, y_val = (
            y_train_outer.iloc[inner_tr_idx],
            y_train_outer.iloc[inner_val_idx],
        )

        # ===== Oversample only the inner-training set. =====
        smoteenn = SMOTEENN(random_state=42)
        X_tr_res, y_tr_res = smoteenn.fit_resample(X_tr, y_tr)

        # ===== train stacking params_joint =====
        model = build_stacking_from_params(params_joint)
        model.fit(X_tr_res, y_tr_res)

        # ===== Predict inner-val ( oversample) =====
        y_val_pred = model.predict(X_val)

        # ===== Metric: recall =====
        rec = recall_score(y_val, y_val_pred, zero_division=0)
        recalls.append(rec)

    return np.mean(recalls)


# =========================================================
# Return best_params_joint and best_inner_recall.
# =========================================================
def tune_joint_params_for_outer_fold(
    X_train_outer,
    y_train_outer,
    n_trials=30,
    n_splits_inner=3,
    random_state_inner=123,
    seed_optuna=42
):
    """
     Optuna :
    - RandomForest base
    - Logistic Regression base
    - SVM base
    - ExtraTrees base
    - Meta Logistic Regression
     (joint tuning)
     Objective: mean recall from inner CV.
    """

    def objective(trial):
        # -------- RandomForest params --------
        rf_n_estimators = trial.suggest_int("rf_n_estimators", 50, 300)
        rf_max_depth = trial.suggest_int("rf_max_depth", 3, 50)
        rf_min_samples_split = trial.suggest_int("rf_min_samples_split", 2, 20)
        rf_min_samples_leaf = trial.suggest_int("rf_min_samples_leaf", 1, 10)
        rf_max_features = trial.suggest_categorical(
            "rf_max_features", ["sqrt", "log2", None]
        )
        rf_bootstrap = trial.suggest_categorical(
            "rf_bootstrap", [True, False]
        )

        # -------- Logistic Regression params --------
        lr_C = trial.suggest_float("lr_C", 1e-4, 1e2, log=True)
        lr_max_iter = trial.suggest_int("lr_max_iter", 100, 2000)

        # -------- SVM params --------
        svm_C = trial.suggest_float("svm_C", 1e-3, 1e3, log=True)
        svm_kernel = trial.suggest_categorical(
            "svm_kernel", ["linear", "rbf", "poly"]
        )

        if svm_kernel in ["rbf", "poly"]:
            svm_gamma = trial.suggest_float("svm_gamma", 1e-4, 1e1, log=True)
        else:
            svm_gamma = None

        if svm_kernel == "poly":
            svm_degree = trial.suggest_int("svm_degree", 2, 5)
        else:
            svm_degree = None

        # -------- ExtraTrees params --------
        et_n_estimators = trial.suggest_int("et_n_estimators", 50, 300)
        et_max_depth = trial.suggest_int("et_max_depth", 3, 50)
        et_min_samples_split = trial.suggest_int("et_min_samples_split", 2, 20)
        et_min_samples_leaf = trial.suggest_int("et_min_samples_leaf", 1, 10)
        et_max_features = trial.suggest_categorical(
            "et_max_features", ["sqrt", "log2", None]
        )
        et_bootstrap = trial.suggest_categorical(
            "et_bootstrap", [True, False]
        )

        # -------- Meta LR params --------
        meta_C = trial.suggest_float("meta_C", 1e-4, 1e2, log=True)
        meta_max_iter = trial.suggest_int("meta_max_iter", 100, 2000)

        params_joint = {
            # RF base
            "rf_n_estimators": rf_n_estimators,
            "rf_max_depth": rf_max_depth,
            "rf_min_samples_split": rf_min_samples_split,
            "rf_min_samples_leaf": rf_min_samples_leaf,
            "rf_max_features": rf_max_features,
            "rf_bootstrap": rf_bootstrap,

            # LR base
            "lr_C": lr_C,
            "lr_max_iter": lr_max_iter,

            # SVM base
            "svm_C": svm_C,
            "svm_kernel": svm_kernel,
            "svm_gamma": svm_gamma,
            "svm_degree": svm_degree,

            # ET base
            "et_n_estimators": et_n_estimators,
            "et_max_depth": et_max_depth,
            "et_min_samples_split": et_min_samples_split,
            "et_min_samples_leaf": et_min_samples_leaf,
            "et_max_features": et_max_features,
            "et_bootstrap": et_bootstrap,

            # Meta LR
            "meta_C": meta_C,
            "meta_max_iter": meta_max_iter
        }

        mean_recall = evaluate_joint_params_inner_cv(
            X_train_outer,
            y_train_outer,
            params_joint,
            n_splits_inner=n_splits_inner,
            random_state_inner=random_state_inner
        )

        # maximize recall
        return mean_recall

    study = optuna.create_study(
        direction="maximize",
        sampler=TPESampler(seed=seed_optuna),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    best_params_joint = study.best_params.copy()

    if "svm_gamma" not in best_params_joint:
        best_params_joint["svm_gamma"] = None
    if "svm_degree" not in best_params_joint:
        best_params_joint["svm_degree"] = None

    best_inner_recall = study.best_value  # Mean recall from inner CV.

    return best_params_joint, best_inner_recall


# =========================================================
# MAIN: Outer nested CV and fold-level reporting.
# =========================================================
def nested_cv_pipeline_stacking_joint(
    X,
    y,
    outer_splits=5,
    inner_splits=3,
    n_trials=30,
    seed_outer=42,
    seed_inner=123,
    seed_optuna=42
):
    """
    outer_splits: Number of folds for final evaluation (e.g., 5).
    inner_splits: Number of folds for tuning (e.g., 3).
    n_trials:        Optuna  outer fold
    """

    outer_kf = StratifiedKFold(
        n_splits=outer_splits,
        shuffle=True,
        random_state=seed_outer
    )

    fold_metrics = []
    fold_params = []

    print("===== Nested CV Stacking (joint Bayesian tuning for RF + LR + SVM + ET + meta-LR) =====\n")

    for fold_idx, (tr_idx, te_idx) in enumerate(outer_kf.split(X, y), start=1):
        X_train_outer = X.iloc[tr_idx] if hasattr(X, "iloc") else X[tr_idx]
        X_test_outer  = X.iloc[te_idx] if hasattr(X, "iloc") else X[te_idx]
        y_train_outer = y.iloc[tr_idx] if hasattr(y, "iloc") else y[tr_idx]
        y_test_outer  = y.iloc[te_idx] if hasattr(y, "iloc") else y[te_idx]

        # -------------------------------------------------
        # -------------------------------------------------
        best_params_fold, best_inner_recall = tune_joint_params_for_outer_fold(
            X_train_outer,
            y_train_outer,
            n_trials=n_trials,
            n_splits_inner=inner_splits,
            random_state_inner=seed_inner,
            seed_optuna=seed_optuna + fold_idx
        )

        fold_params.append(best_params_fold)

        # -------------------------------------------------
        # -------------------------------------------------
        smoteenn = SMOTEENN(random_state=RANDOM_STATE)
        X_train_res, y_train_res = smoteenn.fit_resample(X_train_outer, y_train_outer)

        model_final = build_stacking_from_params(best_params_fold)
        model_final.fit(X_train_res, y_train_res)

        y_pred = model_final.predict(X_test_outer)
        y_proba = model_final.predict_proba(X_test_outer)[:, 1]

        acc  = accuracy_score(y_test_outer, y_pred)
        prec = precision_score(y_test_outer, y_pred, zero_division=0)
        rec  = recall_score(y_test_outer, y_pred, zero_division=0)
        f1   = f1_score(y_test_outer, y_pred, zero_division=0)
        auc  = roc_auc_score(y_test_outer, y_proba)

        cm = confusion_matrix(y_test_outer, y_pred)

        fold_metrics.append({
            "fold": fold_idx,
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "f1": f1,
            "auc": auc,
            "TN": int(cm[0,0]),
            "FP": int(cm[0,1]),
            "FN": int(cm[1,0]),
            "TP": int(cm[1,1]),
            "best_inner_recall": best_inner_recall
        })

        # -------------------------------------------------
        # 3) Display fold-level results.
        # -------------------------------------------------
        print(f"[Outer Fold {fold_idx}]")
        print(f"  Inner-CV tuned recall (objective): {best_inner_recall:.4f}")
        print(f"  Test Performance on this fold:")
        print(f"    Acc={acc:.4f}  Prec={prec:.4f}  Rec={rec:.4f}  "
              f"F1={f1:.4f}  AUC={auc:.4f}")
        print("  Confusion Matrix:")
        print(cm)
        print("  Best Joint Params:")
        print(best_params_fold)
        print("-"*100)

    # -------------------------------------------------
    # 4) Summarize results across all outer folds.
    # -------------------------------------------------
    df_res = pd.DataFrame(fold_metrics)

    print("\n===== Summary over all outer folds =====")
    for m in ["accuracy", "precision", "recall", "f1", "auc"]:
        mean_val = df_res[m].mean()
        std_val  = df_res[m].std(ddof=1)
        print(f"{m.capitalize():<10} mean={mean_val:.4f}  std={std_val:.4f}")

    print("\nPer-fold confusion / best params:")
    for row in fold_metrics:
        print(f" Fold {row['fold']}: "
              f"TN={row['TN']} FP={row['FP']} FN={row['FN']} TP={row['TP']}")
        print(f"  Params: {fold_params[row['fold']-1]}")
        print()

    summary_results = {
        "Accuracy":  (df_res["accuracy"].mean(),  df_res["accuracy"].std(ddof=1)),
        "Precision": (df_res["precision"].mean(), df_res["precision"].std(ddof=1)),
        "Recall":    (df_res["recall"].mean(),    df_res["recall"].std(ddof=1)),
        "F1":        (df_res["f1"].mean(),        df_res["f1"].std(ddof=1)),
        "AUC":       (df_res["auc"].mean(),       df_res["auc"].std(ddof=1)),
    }

    return summary_results, fold_metrics, fold_params


# =========================================================
# Usage:
# X = df.drop("target", axis=1)
# y = df["target"]
#
# Then call:
summary_results, fold_metrics, params_each_fold = nested_cv_pipeline_stacking_joint(
      X, y,
      outer_splits=5,
      inner_splits=3,
      n_trials=100,
      seed_outer=42,
      seed_inner=123,
      seed_optuna=42
  )
#
# fold_metrics     -> fold-level performance and confusion matrix. counts
# =========================================================
